<a href="https://colab.research.google.com/github/mic006016/geo-referencing-ai-pipeline/blob/main/GeoAI_YOLOv8_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Ultralytics 라이브러리 설치
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 46.7/235.7 GB disk)


In [2]:
import os
import shutil
import cv2
import glob

from ultralytics import YOLO

In [3]:
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

# 2. 드라이브의 zip 파일을 코랩의 로컬 가상 디스크(/content)로 복사
zip_path = "/content/drive/MyDrive/yolo_dataset.zip"
local_zip_path = "/content/yolo_dataset.zip"

print("데이터 복사 중...")
shutil.copy(zip_path, local_zip_path)
print("복사 완료!")

# 3. 로컬에서 압축 해제 (-q 옵션으로 출력 생략하여 브라우저 렉 방지)
!unzip -q {local_zip_path} -d /content/yolo_dataset
print("압축 해제 완료!")

Mounted at /content/drive
데이터 복사 중...
복사 완료!
압축 해제 완료!


In [ ]:
from ultralytics import YOLO

# 1. 모델 로드
model = YOLO('yolov8s.pt')

# 2. 전이 학습 시작
results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=50,
    imgsz=1024,            # v3: 해상도 증가
    batch=64,
    device=0,              # 0번 GPU 사용
    workers=8,             # 데이터 로딩에 사용할 CPU 워커 수
    amp=True,              # 자동 혼합 정밀도 (연산 속도 가속)
    project='geo_ai',      # 결과물이 저장될 폴더명
    name='land_cover_v3',
    save=True,             # 매 에포크마다 가중치 저장
    patience=10,           # 10 에포크 동안 성능 향상이 없으면 조기 종료
)

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=land_cover_v

In [5]:
#3. 테스트셋 평가
metrics = model.val(
    data='/content/yolo_dataset/dataset.yaml',
    split='test',          # 훈련에 관여하지 않은 테스트 데이터 지정
    project='geo_ai',
    name='land_cover_v3_test'
)

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2137.2±469.3 MB/s, size: 109.2 KB)
val: Scanning /content/yolo_dataset/test/labels... 4465 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4465/4465 1.5Kit/s 3.0s
val: New cache created: /content/yolo_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 280/280 7.1it/s 39.6s
                   all       4465      62734      0.851      0.805      0.864      0.705
              Building       2971      22477      0.848      0.754      0.852       0.68
            GreenHouse       2182      11392      0.829       0.85      0.874      0.744
                  Road       1543       2249      0.877      0.856      0.893      0.705
             RicePaddy       2803      14740      0.91

In [6]:
from google.colab import files

# 1. geo_ai 폴더를 results_v1.zip으로 압축 (-q 옵션으로 로그 생략)
!zip -r -q /content/results_v3.zip /content/runs/detect/geo_ai/land_cover_v3_test

print("✅ 압축 완료! PC로 다운로드를 시작합니다.")

# 2. 브라우저를 통해 내 로컬 PC로 다운로드
files.download('/content/results_v3.zip')

✅ 압축 완료! PC로 다운로드를 시작합니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>